# Task 2.2 - Reproduce a Core Contribution


**Reproduction target:** UCR Suite exact DTW subsequence search with a cascade of lower bounds (LB_KimFL -> LB_Keogh -> LB_Keogh2) and early abandoning.  
**Evaluation metric:** wall-clock search time and number of DTW computations (the paper reports time-to-search for fixed query length).


In [1]:

import numpy as np
import time

data = np.load("partB/data/synthetic_series.npz")
T = data["T"]
Q = data["Q"]

SEED = 42
m = len(Q)
r = max(1, int(0.05 * m))  # 5% warping window

def z_normalize(ts):
    mean = ts.mean()
    std = ts.std()
    if std < 1e-8:
        return ts - mean
    return (ts - mean) / std

def envelope(ts, r):
    n = len(ts)
    upper = np.empty(n)
    lower = np.empty(n)
    for i in range(n):
        start = max(0, i - r)
        end = min(n, i + r + 1)
        window = ts[start:end]
        upper[i] = window.max()
        lower[i] = window.min()
    return upper, lower

def lb_kim_fl(q, c):
    return (q[0] - c[0]) ** 2 + (q[-1] - c[-1]) ** 2

def lb_keogh(c, upper, lower, bsf):
    s = 0.0
    for i, v in enumerate(c):
        if v > upper[i]:
            d = (v - upper[i]) ** 2
        elif v < lower[i]:
            d = (v - lower[i]) ** 2
        else:
            d = 0.0
        s += d
        if s > bsf:
            return s
    return s

def dtw_distance(q, c, r, bsf):
    m = len(q)
    w = r
    INF = 1e18
    dtw = np.full((m + 1, m + 1), INF)
    dtw[0, 0] = 0.0
    for i in range(1, m + 1):
        start = max(1, i - w)
        end = min(m, i + w)
        row_min = INF
        for j in range(start, end + 1):
            cost = (q[i - 1] - c[j - 1]) ** 2
            dtw[i, j] = cost + min(dtw[i - 1, j], dtw[i, j - 1], dtw[i - 1, j - 1])
            row_min = min(row_min, dtw[i, j])
        if row_min > bsf:
            return row_min
    return dtw[m, m]


Loaded dataset and defined core functions.


**Explanation:** This block loads the dataset and defines Z-normalization (Section 4.2.1), envelope construction for LB_Keogh (Section 4.1.2), and constrained DTW with early abandoning (Section 4.1.4).

In [1]:

def ucr_dtw_search(T, Q, r):
    m = len(Q)
    q = z_normalize(Q)
    U, L = envelope(q, r)
    bsf = float('inf')
    best_idx = -1
    stats = {"total": 0, "pruned_kim": 0, "pruned_keogh": 0, "pruned_keogh2": 0, "dtw_calls": 0}

    for i in range(0, len(T) - m + 1):
        stats["total"] += 1
        c = z_normalize(T[i:i + m])

        if lb_kim_fl(q, c) >= bsf:
            stats["pruned_kim"] += 1
            continue

        if lb_keogh(c, U, L, bsf) >= bsf:
            stats["pruned_keogh"] += 1
            continue

        Uc, Lc = envelope(c, r)
        if lb_keogh(q, Uc, Lc, bsf) >= bsf:
            stats["pruned_keogh2"] += 1
            continue

        stats["dtw_calls"] += 1
        d = dtw_distance(q, c, r, bsf)
        if d < bsf:
            bsf = d
            best_idx = i

    return bsf, best_idx, stats

def naive_search(T, Q, r):
    m = len(Q)
    q = z_normalize(Q)
    bsf = float('inf')
    best_idx = -1
    dtw_calls = 0
    for i in range(0, len(T) - m + 1):
        c = z_normalize(T[i:i + m])
        dtw_calls += 1
        d = dtw_distance(q, c, r, bsf)
        if d < bsf:
            bsf = d
            best_idx = i
    return bsf, best_idx, dtw_calls


Defined UCR-DTW search and naive DTW baseline.


**Explanation:** This block implements the paper's cascade (LB_KimFL -> LB_Keogh -> LB_Keogh2 -> DTW) as described in Section 4.1-4.2 and Table 1.

In [1]:

def timed(fn, runs=3):
    times = []
    last = None
    for _ in range(runs):
        start = time.perf_counter()
        last = fn()
        times.append(time.perf_counter() - start)
    return float(np.mean(times)), last

naive_time, (naive_bsf, naive_idx, naive_calls) = timed(lambda: naive_search(T, Q, r))
ucr_time, (ucr_bsf, ucr_idx, ucr_stats) = timed(lambda: ucr_dtw_search(T, Q, r))

print(f"Naive DTW: time={naive_time:.4f}s, best_idx={naive_idx}, dist={naive_bsf:.4f}, calls={naive_calls}")
print(f"UCR-DTW:  time={ucr_time:.4f}s, best_idx={ucr_idx}, dist={ucr_bsf:.4f}, calls={ucr_stats['dtw_calls']}")
print(f"Pruned: kim={ucr_stats['pruned_kim']}, keogh={ucr_stats['pruned_keogh']}, keogh2={ucr_stats['pruned_keogh2']}")


Naive DTW: time=0.3364s, best_idx=700, dist=0.5053, calls=1873
UCR-DTW:  time=0.0791s, best_idx=700, dist=0.5053, calls=48
Pruned: kim=1108, keogh=687, keogh2=30


**Interpretation:** The cascade finds the correct location with far fewer DTW calls, which matches the pruning strategy in Sections 4.1 to 4.2 and the UCR-DTW outline in Table 1. The best-so-far update enables early abandoning so most candidates stop at lower bounds. This behavior mirrors the paper's reported speedups for fixed query length.